# Análise Exploratória de Dados — Consumo de Energia (Sines)

**Objetivo:** caracterizar o consumo de energia elétrica em **Sines (código postal 7520)** e relacioná-lo
com o consumo, produção e injeção de energia a nível nacional.

**Dados (públicos, E-REDES):**
- `data/codigos_postais/*.xlsx` — consumo horário por código postal (incluindo 7520 — Sines)
- `data/consumo-total-nacional.xlsx` — consumo total nacional (kWh)
- `data/energia-injetada-na-rede-de-distribuicao.xlsx` — energia injetada na rede
- `data/energia-produzida-total-nacional.xlsx` — produção nacional (Eólica, Hídrica, Fotovoltaica, ...)

> **Nota sobre escalas:** o consumo de **Sines** é da ordem de poucos MWh/hora,
> enquanto o **consumo nacional** está na ordem dos GWh/hora.
> Por isso, gráficos comparativos usam **dois eixos Y (twin axis)** ou **normalização**.

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# Estilo gráfico moderno e legível
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (12, 5),
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)
print("Diretório de dados:", os.path.abspath(DATA_DIR))

## 1. Carregar consumo por código postal

Os ficheiros vêm da E-REDES com colunas:
`Data/Hora`, `Código Postal`, `Energia ativa (kWh)`, `Dia`, `Hora`, `Dia da Semana`.

> Aqui colocamos **todos os ficheiros .xlsx** dos códigos postais numa subpasta
> `data/codigos_postais/`. Se preferires, podes colocá-los diretamente em `data/`
> e ajustar o caminho `PASTA_CP`.

In [ ]:
PASTA_CP = os.path.join(DATA_DIR, "codigos_postais")

# Fallback: se a subpasta não existir, usa todos os .xlsx que estão em data/ excepto os 3 ficheiros nacionais
ficheiros_nacionais = {
    "consumo-total-nacional.xlsx",
    "energia-injetada-na-rede-de-distribuicao.xlsx",
    "energia-produzida-total-nacional.xlsx",
}

if os.path.isdir(PASTA_CP):
    arquivos = [os.path.join(PASTA_CP, f) for f in os.listdir(PASTA_CP) if f.endswith(".xlsx")]
else:
    arquivos = [
        os.path.join(DATA_DIR, f)
        for f in os.listdir(DATA_DIR)
        if f.endswith(".xlsx") and f not in ficheiros_nacionais
    ]

print(f"Encontrados {len(arquivos)} ficheiros de códigos postais.")
arquivos[:5]

In [ ]:
dfs = []
for caminho in arquivos:
    try:
        d = pd.read_excel(caminho)
        dfs.append(d)
    except Exception as e:
        print(f"Erro a ler {caminho}: {e}")

df_codigo = pd.concat(dfs, ignore_index=True)
print("Shape consumo por código postal (long):", df_codigo.shape)
df_codigo.head()

In [ ]:
# Manter apenas as colunas relevantes e converter para datetime
df_codigo = df_codigo[["Data/Hora", "Código Postal", "Energia ativa (kWh)"]].copy()
df_codigo["Data/Hora"] = pd.to_datetime(df_codigo["Data/Hora"], errors="coerce")
df_codigo["Energia ativa (kWh)"] = pd.to_numeric(df_codigo["Energia ativa (kWh)"], errors="coerce")

# Diagnóstico de qualidade
print("Valores em falta por coluna:\n", df_codigo.isna().sum())
print("\nPeríodo:", df_codigo["Data/Hora"].min(), "->", df_codigo["Data/Hora"].max())
print("Códigos postais únicos:", sorted(df_codigo["Código Postal"].unique()))

In [ ]:
# Pivot wide: uma coluna por código postal.
# Importante: aggfunc='mean' (caso haja duplicados) e SEM fill_value (preservar NaN para diagnóstico)
df_codigo_pivot = (
    df_codigo
    .pivot_table(index="Data/Hora", columns="Código Postal",
                 values="Energia ativa (kWh)", aggfunc="mean")
    .sort_index()
)
df_codigo_pivot.columns = [int(c) for c in df_codigo_pivot.columns]
df_codigo_pivot = df_codigo_pivot.reindex(columns=sorted(df_codigo_pivot.columns))

print("Shape (wide):", df_codigo_pivot.shape)
print("\n% de NaN por código postal:")
print((df_codigo_pivot.isna().mean() * 100).round(2))

df_codigo_pivot.head()

## 2. Carregar consumo total nacional

In [ ]:
df_consumo = pd.read_excel(os.path.join(DATA_DIR, "consumo-total-nacional.xlsx"))
df_consumo = df_consumo.drop(columns=["Dia", "Mês", "Ano", "Data", "Hora"], errors="ignore")
df_consumo["Data/Hora"] = pd.to_datetime(df_consumo["Data/Hora"], errors="coerce")
df_consumo = df_consumo.rename(columns={"Total (kWh)": "Total_Consumido_kWh"})
print(df_consumo.shape)
df_consumo.head()

## 3. Carregar energia injetada na rede

In [ ]:
df_injetada = pd.read_excel(os.path.join(DATA_DIR, "energia-injetada-na-rede-de-distribuicao.xlsx"))
df_injetada = df_injetada.drop(columns=["Dia", "Mês", "Ano", "Data", "Hora"], errors="ignore")
df_injetada["Data/Hora"] = pd.to_datetime(df_injetada["Data/Hora"], errors="coerce")
print(df_injetada.shape)
df_injetada.head()

## 4. Carregar energia produzida (mix energético nacional)

In [ ]:
df_produzida = pd.read_excel(os.path.join(DATA_DIR, "energia-produzida-total-nacional.xlsx"))
df_produzida = df_produzida.drop(columns=["Dia", "Mês", "Ano", "Data", "Hora"], errors="ignore")
df_produzida["Data/Hora"] = pd.to_datetime(df_produzida["Data/Hora"], errors="coerce")
df_produzida = df_produzida.rename(columns={"Total (kWh)": "Total_Produzido_kWh"})
print(df_produzida.shape)
df_produzida.head()

## 5. Junção dos 4 datasets

Usamos `merge` em `Data/Hora` (chave horária). **Mantemos os NaN visíveis** para inspeção.

In [ ]:
# Garantir mesmo tipo de datetime em todos
for d in (df_codigo_pivot, df_consumo, df_injetada, df_produzida):
    if d.index.name == "Data/Hora":
        d.index = pd.to_datetime(d.index)

df_codigo_pivot_reset = df_codigo_pivot.reset_index()

df_merged = (
    df_codigo_pivot_reset
    .merge(df_consumo, on="Data/Hora", how="inner")
    .merge(df_injetada, on="Data/Hora", how="inner")
    .merge(df_produzida, on="Data/Hora", how="inner")
    .sort_values("Data/Hora")
    .reset_index(drop=True)
)

print("Shape final:", df_merged.shape)
print("Período:", df_merged["Data/Hora"].min(), "->", df_merged["Data/Hora"].max())
df_merged.head()

In [ ]:
# Diagnóstico de valores em falta no dataset combinado
nan_pct = (df_merged.isna().mean() * 100).round(2).sort_values(ascending=False)
print("% de NaN por coluna (top 15):")
print(nan_pct.head(15))

In [ ]:
# Tratamento de NaN nas séries horárias:
# - Para Sines (7520) e para colunas numéricas usamos interpolação temporal (curta) e depois ffill/bfill
df_merged = df_merged.set_index("Data/Hora").sort_index()
num_cols = df_merged.select_dtypes(include=[np.number]).columns

df_merged[num_cols] = (
    df_merged[num_cols]
    .interpolate(method="time", limit=6)   # até 6h consecutivas
    .ffill(limit=24)
    .bfill(limit=24)
)
df_merged = df_merged.dropna(subset=[7520])  # exigimos o target válido
df_merged = df_merged.reset_index()
print("Shape após interpolação:", df_merged.shape)
print("NaN restantes:", int(df_merged.isna().sum().sum()))

## 6. Variáveis de calendário

In [ ]:
df_merged["Ano"]        = df_merged["Data/Hora"].dt.year
df_merged["Mês"]        = df_merged["Data/Hora"].dt.month
df_merged["Dia"]        = df_merged["Data/Hora"].dt.day
df_merged["Hora"]       = df_merged["Data/Hora"].dt.hour
df_merged["Dia_Semana"] = df_merged["Data/Hora"].dt.dayofweek   # 0=Seg ... 6=Dom
df_merged["Semana"]     = df_merged["Data/Hora"].dt.isocalendar().week.astype(int)

df_merged.head()

## 7. Estatística descritiva (Sines + variáveis nacionais)

> **Atenção às unidades:** todos os valores estão em **kWh**.
> Sines (7520) é um único código postal → valores pequenos.
> Total Consumido / Total Produzido são valores agregados nacionais → valores muito maiores.

In [ ]:
codigos_postais_existentes = [c for c in [2800, 2830, 2860, 2870, 2890, 2900, 2950, 2970, 7520, 7570, 7580]
                              if c in df_merged.columns]

cols_resumo = codigos_postais_existentes + ["Total_Consumido_kWh", "Total_Produzido_kWh"]
df_merged[cols_resumo].describe().T.round(2)

## 8. Série temporal — Sines vs Nacional (dois eixos)

Como as escalas são muito diferentes, usamos **eixos Y duplos** para comparar a dinâmica.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 5))
ax2 = ax1.twinx()

ax1.plot(df_merged["Data/Hora"], df_merged[7520], color="#1f77b4", linewidth=0.8, label="Sines (7520)")
ax2.plot(df_merged["Data/Hora"], df_merged["Total_Consumido_kWh"], color="#d62728",
         linewidth=0.8, alpha=0.7, label="Total Nacional")

ax1.set_ylabel("Consumo Sines (kWh)", color="#1f77b4")
ax2.set_ylabel("Consumo Nacional (kWh)", color="#d62728")
ax1.tick_params(axis="y", labelcolor="#1f77b4")
ax2.tick_params(axis="y", labelcolor="#d62728")
ax1.set_xlabel("Data")
ax1.set_title("Consumo horário — Sines (7520) vs Consumo Total Nacional")
ax1.xaxis.set_major_locator(mdates.AutoDateLocator())
ax1.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax1.xaxis.get_major_locator()))
ax2.grid(False)

# Legenda combinada
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
plt.tight_layout()
plt.show()

## 9. Padrão sazonal — média mensal (normalizada)

Para comparar perfis (e não magnitudes), normalizamos cada série (z-score).

In [ ]:
monthly = df_merged.groupby("Mês")[[7520, "Total_Consumido_kWh", "Total_Produzido_kWh"]].mean()
monthly_z = (monthly - monthly.mean()) / monthly.std()

fig, ax = plt.subplots(figsize=(11, 4.5))
for col, label in [(7520, "Sines (7520)"),
                   ("Total_Consumido_kWh", "Consumo Nacional"),
                   ("Total_Produzido_kWh", "Produção Nacional")]:
    ax.plot(monthly_z.index, monthly_z[col], marker="o", label=label, linewidth=2)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"])
ax.set_ylabel("Z-score (perfil sazonal)")
ax.set_title("Perfil sazonal mensal (normalizado) — Sines vs Nacional")
ax.legend()
plt.tight_layout(); plt.show()

## 10. Perfil horário do consumo em Sines (boxplot)

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
sns.boxplot(data=df_merged, x="Hora", y=7520, ax=ax,
            color="#4C72B0", fliersize=2, linewidth=0.8)
ax.set_title("Distribuição horária do consumo — Sines (7520)")
ax.set_xlabel("Hora do dia")
ax.set_ylabel("Consumo (kWh)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=df_merged, x="Dia_Semana", y=7520, ax=ax,
            order=[0,1,2,3,4,5,6], color="#55A868", fliersize=2, linewidth=0.8)
ax.set_xticklabels(["Seg","Ter","Qua","Qui","Sex","Sáb","Dom"])
ax.set_title("Distribuição diária do consumo — Sines (7520)")
ax.set_xlabel("Dia da semana")
ax.set_ylabel("Consumo (kWh)")
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
sns.boxplot(data=df_merged, x="Mês", y=7520, ax=ax,
            color="#C44E52", fliersize=2, linewidth=0.8)
ax.set_title("Distribuição mensal do consumo — Sines (7520)")
ax.set_xlabel("Mês")
ax.set_ylabel("Consumo (kWh)")
plt.tight_layout(); plt.show()

## 11. Histograma de Sines + curva de densidade

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
sns.histplot(df_merged[7520], bins=80, kde=True, color="#1f77b4", ax=ax)
ax.set_title("Distribuição do consumo horário em Sines (7520)")
ax.set_xlabel("Consumo (kWh)"); ax.set_ylabel("Frequência")
plt.tight_layout(); plt.show()

## 12. Correlações entre Sines e o resto do sistema

In [ ]:
colunas_producao = [c for c in [
    "Cogeração (kWh)", "Eólica (kWh)", "Fotovoltaica (kWh)", "Hídrica (kWh)",
    "Outras Tecnologias (kWh)", "Rede Distribuição (kWh)", "Mercado (kWh)",
    "Regime Especial (kWh)"] if c in df_merged.columns]

cols_corr = codigos_postais_existentes + ["Total_Consumido_kWh", "Total_Produzido_kWh"] + colunas_producao
corr = df_merged[cols_corr].corr()

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(corr, cmap="RdBu_r", center=0, annot=True, fmt=".2f",
            annot_kws={"size": 7}, square=False, cbar_kws={"shrink": .8}, ax=ax)
ax.set_title("Matriz de correlação — códigos postais, consumo e produção")
plt.tight_layout(); plt.show()

print("\nCorrelações de Sines (7520) com restantes variáveis (ordenadas):")
print(corr[7520].drop(7520).sort_values(ascending=False).round(3))

## 13. Outliers — diagnóstico (sem remoção agressiva)

Em vez de descartar via z-score, **inspecionamos** e marcamos.
Mantemos as observações para os modelos (LSTM/GRU/XGBoost lidam bem com picos quando há lags).

In [ ]:
z = (df_merged[7520] - df_merged[7520].mean()) / df_merged[7520].std()
df_merged["Outlier_z3"] = (z.abs() > 3).astype(int)
print(f"Outliers |z|>3: {df_merged['Outlier_z3'].sum()}  ({df_merged['Outlier_z3'].mean()*100:.2f}%)")

## 14. Feature engineering para modelação

- **Estação do ano** (Inverno/Primavera/Verão/Outono) → one-hot
- **Fim de semana** (0/1)
- **Variáveis cíclicas** (sin/cos) para Hora, Dia da Semana e Mês

In [ ]:
def obter_estacao(data):
    ano = data.year
    prim   = pd.Timestamp(year=ano, month=3, day=21)
    verao  = pd.Timestamp(year=ano, month=6, day=21)
    outono = pd.Timestamp(year=ano, month=9, day=23)
    inv    = pd.Timestamp(year=ano, month=12, day=21)
    if   prim   <= data < verao:  return "Primavera"
    elif verao  <= data < outono: return "Verão"
    elif outono <= data < inv:    return "Outono"
    else:                         return "Inverno"

df_merged["Estacao"] = df_merged["Data/Hora"].apply(obter_estacao)
df_merged = pd.get_dummies(df_merged, columns=["Estacao"], prefix="Estacao", dtype=int)
df_merged["Fim_de_Semana"] = df_merged["Dia_Semana"].isin([5, 6]).astype(int)

# Cíclicas
df_merged["Hora_sin"]       = np.sin(2*np.pi*df_merged["Hora"]/24)
df_merged["Hora_cos"]       = np.cos(2*np.pi*df_merged["Hora"]/24)
df_merged["Dia_Semana_sin"] = np.sin(2*np.pi*df_merged["Dia_Semana"]/7)
df_merged["Dia_Semana_cos"] = np.cos(2*np.pi*df_merged["Dia_Semana"]/7)
df_merged["Mes_sin"]        = np.sin(2*np.pi*(df_merged["Mês"]-1)/12)
df_merged["Mes_cos"]        = np.cos(2*np.pi*(df_merged["Mês"]-1)/12)

df_merged.head(3)

## 15. Guardar dataset final para os modelos

> **IMPORTANTE:** NÃO convertemos o dataframe inteiro para `int`
> (isso corromperia os valores em kWh e a coluna `Data/Hora`).
> Mantemos os tipos nativos.

In [ ]:
out_path = os.path.join(DATA_DIR, "dataset_completo_sines.xlsx")
df_merged.to_excel(out_path, index=False)
print("Guardado em:", out_path)
print("Shape:", df_merged.shape)
print("Colunas:", df_merged.columns.tolist())

---
### Conclusões da EDA

- O consumo em **Sines (7520)** tem forte **periodicidade horária e semanal** e variação **sazonal** marcada.
- A correlação com o **consumo total nacional** é moderada (perfil parecido, mas Sines é dominado por consumo industrial local — porto/refinaria).
- Os ficheiros de produção nacional (eólica, hídrica, etc.) trazem informação útil para o modelo.
- Não foram removidos outliers de forma agressiva — apenas marcados; LSTM/GRU lidam bem com lags.

O ficheiro `data/dataset_completo_sines.xlsx` é o input dos notebooks **02_LSTM**, **03_GRU** e **04_XGBoost**.